# Xarray-Spatial Cost Distance: Weighted proximity and travel cost

Straight-line distance works until terrain gets in the way. A point 2 km behind a ridge might take three times longer to reach than one 2 km down the valley. `cost_distance` computes the minimum accumulated traversal cost from source cells through a friction surface, so the output actually reflects what the ground is like between A and B.

### What you'll build

1. Generate terrain and derive a friction surface from slope
2. Compute cost distance from a single source point
3. Compare cost distance with straight-line proximity
4. Route around impassable barriers
5. Map reachable area within a travel budget using `max_cost`
6. Split the landscape into cost-based territories for two sources

![Cost distance routing around a barrier](images/cost_distance_barrier_preview.png)

[Terrain](#Terrain) · [Friction surface](#Friction-surface) · [Single-source cost distance](#Single-source-cost-distance) · [Cost distance vs proximity](#Cost-distance-vs-proximity) · [Barriers](#Barriers) · [Travel budget](#Travel-budget) · [Multiple sources](#Multiple-sources)

Standard imports.

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr

import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

import xrspatial

## Terrain

Synthetic terrain from xrspatial's procedural generator. We compute hillshade and slope up front since every section uses them.

In [ ]:
W = 800
H = 600

terrain = xr.DataArray(np.zeros((H, W)))
terrain = terrain.xrs.generate_terrain(
    x_range=(-250, 250), y_range=(-250, 250),
    zfactor=6000, warp_strength=0.4,
)

hillshade = terrain.xrs.hillshade(boundary='nearest')
slope = terrain.xrs.slope()

terrain.plot.imshow(cmap='gray', size=7.5, aspect=W/H, add_colorbar=False)

Raw elevation in grayscale. Adding hillshade brings out the ridgelines and valleys.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7.5))
hillshade.plot.imshow(ax=ax, cmap='gray', add_colorbar=False)
terrain.plot.imshow(ax=ax, cmap='terrain', alpha=128/255, add_colorbar=False)
ax.set_axis_off()

## Friction surface

Friction represents how costly each cell is to cross. Here we derive it from slope: flat ground gets friction 1, and cost scales linearly with steepness. A 50-degree slope is 11 times harder to cross than flat ground.

The plot shows friction overlaid on hillshade. Ridgelines light up red; valleys stay yellow.

In [ ]:
friction = 1 + slope / 5

fig, ax = plt.subplots(figsize=(10, 7.5))
hillshade.plot.imshow(ax=ax, cmap='gray', add_colorbar=False)
friction.plot.imshow(ax=ax, cmap='YlOrRd', alpha=200/255, add_colorbar=True,
                     cbar_kwargs={'label': 'Friction', 'shrink': 0.7})
ax.set_axis_off()

<div class="alert alert-block alert-info">
<b>Friction values are relative.</b> What matters is the ratio between cells, not the absolute numbers. Doubling all friction values doubles all output costs but doesn't change which path is cheapest. Scale your friction so the numbers mean something for your use case (e.g., minutes per meter, fuel per cell).
</div>

## Single-source cost distance

`cost_distance` takes a source raster (non-zero cells are starting points) and a friction surface, then finds the cheapest path from every cell back to the nearest source using [Dijkstra's algorithm](https://en.wikipedia.org/wiki/Dijkstra%27s_algorithm). The output is accumulated cost, not distance.

We place one source in the lowlands and see how cost spreads through the terrain.

In [ ]:
src_row, src_col = 450, 150
source = xr.zeros_like(terrain)
source.values[src_row, src_col] = 1.0

cd = source.xrs.cost_distance(friction)

fig, ax = plt.subplots(figsize=(10, 7.5))
hillshade.plot.imshow(ax=ax, cmap='gray', add_colorbar=False)
cd.plot.imshow(ax=ax, cmap='inferno', alpha=200/255, add_colorbar=True,
               cbar_kwargs={'label': 'Accumulated cost', 'shrink': 0.7})
ax.plot(terrain.x.values[src_col], terrain.y.values[src_row],
        'w*', markersize=15, zorder=5)
ax.set_axis_off()

Cost contours stretch along valleys where traversal is cheap. They compress against ridgelines. A point on the far side of a ridge costs more to reach than one at the same straight-line distance in the same valley.

## Cost distance vs proximity

`proximity` measures straight-line Euclidean distance and ignores terrain. `cost_distance` accumulates friction along the cheapest grid path. On flat ground with uniform friction the two agree. On mountains they diverge.

In [ ]:
prox = source.xrs.proximity()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

ax1.set_title('Proximity (Euclidean)', fontsize=13)
hillshade.plot.imshow(ax=ax1, cmap='gray', add_colorbar=False)
prox.plot.imshow(ax=ax1, cmap='inferno', alpha=200/255, add_colorbar=True,
                 cbar_kwargs={'label': 'Distance', 'shrink': 0.7})
ax1.plot(terrain.x.values[src_col], terrain.y.values[src_row],
         'w*', markersize=12, zorder=5)
ax1.set_axis_off()

ax2.set_title('Cost distance', fontsize=13)
hillshade.plot.imshow(ax=ax2, cmap='gray', add_colorbar=False)
cd.plot.imshow(ax=ax2, cmap='inferno', alpha=200/255, add_colorbar=True,
               cbar_kwargs={'label': 'Accumulated cost', 'shrink': 0.7})
ax2.plot(terrain.x.values[src_col], terrain.y.values[src_row],
         'w*', markersize=12, zorder=5)
ax2.set_axis_off()

plt.tight_layout()

Proximity gives smooth concentric circles. Cost distance doesn't: the contours warp through valleys and bunch up against ridges. Near the source the two mostly agree, since the terrain there is locally flat.

<div class="alert alert-block alert-warning">
<b>Grid discretization.</b> <code>cost_distance</code> runs Dijkstra on an 8-connected grid. The shortest 8-connected path overshoots true Euclidean distance for off-axis directions by up to (√2 − 1) per cell step. For most practical rasters this error is small, but it's worth knowing if you compare results against analytic solutions.
</div>

## Barriers

Setting friction to NaN makes cells impassable. `cost_distance` routes around them. If a region is completely enclosed by barriers, it gets NaN (unreachable).

Here we drop a horizontal barrier across the terrain with one crossing point, like a river gorge with a single bridge.

In [ ]:
friction_barrier = friction.copy()
friction_barrier.values[290:310, :] = np.nan
friction_barrier.values[296:304, 380:420] = friction.values[296:304, 380:420]

cd_barrier = source.xrs.cost_distance(friction_barrier)

barrier_cells = xr.DataArray(
    np.where(np.isnan(friction_barrier.values) & ~np.isnan(friction.values), 1.0, np.nan),
    dims=terrain.dims, coords=terrain.coords,
)

fig, ax = plt.subplots(figsize=(10, 7.5))
hillshade.plot.imshow(ax=ax, cmap='gray', add_colorbar=False)
cd_barrier.plot.imshow(ax=ax, cmap='inferno', alpha=200/255, add_colorbar=True,
                       cbar_kwargs={'label': 'Accumulated cost', 'shrink': 0.7})
barrier_cells.plot.imshow(ax=ax, cmap=ListedColormap(['steelblue']),
                          alpha=0.8, add_colorbar=False)
ax.plot(terrain.x.values[src_col], terrain.y.values[src_row],
        'w*', markersize=15, zorder=5)
ax.legend(handles=[Patch(facecolor='steelblue', alpha=0.8, label='Barrier (NaN friction)')],
          loc='lower right', fontsize=11, framealpha=0.9)
ax.set_axis_off()

Every path to the far side has to squeeze through the crossing point. The contours on the far side are lopsided because of the detour, and cells right behind the barrier wind up costing far more than their straight-line distance would suggest.

## Travel budget

The `max_cost` parameter caps the search. Cells whose cheapest path exceeds the budget get NaN. This answers "what's reachable within a given cost?" It also matters for performance with Dask on large rasters, since `max_cost` bounds the overlap region each chunk needs.

In [ ]:
cd_budget = source.xrs.cost_distance(friction, max_cost=400)

fig, ax = plt.subplots(figsize=(10, 7.5))
hillshade.plot.imshow(ax=ax, cmap='gray', add_colorbar=False)
cd_budget.plot.imshow(ax=ax, cmap='inferno', alpha=200/255, add_colorbar=True,
                      cbar_kwargs={'label': 'Accumulated cost', 'shrink': 0.7})
ax.plot(terrain.x.values[src_col], terrain.y.values[src_row],
        'w*', markersize=15, zorder=5)
ax.set_axis_off()

The reachable zone isn't a circle. It reaches farther through valleys and stops short at ridgelines. Gray is everything outside the budget.

## Multiple sources

With several source cells, `cost_distance` finds the cheapest path to whichever source is nearest by cost. To see which source claims which area, compute cost distance from each source separately and compare.

Unlike `allocation` (which uses straight-line distance), cost-based territories warp around the terrain. A source in a valley claims ground along the valley floor even when a hilltop is geometrically closer to the other source.

In [ ]:
src_a_row, src_a_col = 450, 150
src_b_row, src_b_col = 100, 650

source_a = xr.zeros_like(terrain)
source_a.values[src_a_row, src_a_col] = 1.0

source_b = xr.zeros_like(terrain)
source_b.values[src_b_row, src_b_col] = 1.0

cd_a = source_a.xrs.cost_distance(friction)
cd_b = source_b.xrs.cost_distance(friction)

valid = cd_a.notnull() & cd_b.notnull()
territory = xr.where(cd_a < cd_b, 1.0, 2.0)
territory = territory.where(valid)

fig, ax = plt.subplots(figsize=(10, 7.5))
hillshade.plot.imshow(ax=ax, cmap='gray', add_colorbar=False)
territory.plot.imshow(ax=ax, cmap=ListedColormap(['darkorange', 'steelblue']),
                      alpha=200/255, add_colorbar=False)
ax.plot(terrain.x.values[src_a_col], terrain.y.values[src_a_row],
        'w*', markersize=15, zorder=5)
ax.plot(terrain.x.values[src_b_col], terrain.y.values[src_b_row],
        'w*', markersize=15, zorder=5)
ax.legend(handles=[Patch(facecolor='darkorange', alpha=0.78, label='Closer to source A'),
                   Patch(facecolor='steelblue', alpha=0.78, label='Closer to source B')],
          loc='lower right', fontsize=11, framealpha=0.9)
ax.set_axis_off()

The territory boundary follows ridgelines instead of cutting straight across. Steep terrain is expensive, so it works like a natural fence between the two sources.

<div class="alert alert-block alert-warning">
<b>Output units depend on input units.</b> Cost distance multiplies friction by geometric step length at each cell. If friction is in seconds per meter and cells are 30 m wide, the output is in seconds. If friction is unitless and coordinates are in degrees, the output is in degree-friction units, which is not physically meaningful. Projected coordinates in meters with friction in consistent units give interpretable results.
</div>

### References

- [Understanding cost distance analysis](https://pro.arcgis.com/en/pro-app/latest/tool-reference/spatial-analyst/understanding-cost-distance-analysis.htm), Esri
- [r.cost](https://grass.osgeo.org/grass-stable/manuals/r.cost.html), GRASS GIS
- Dijkstra, E. W. (1959). A note on two problems in connexion with graphs. *Numerische Mathematik*, 1(1), 269-271.